# 15 — ICI Specificity of CIDER T-cell Score

Tests whether the CIDER T-cell response and survival associations are specific
to immune checkpoint inhibitor (ICI) therapy.

**Covariates:** `cohort_pan` (tumor type) + `scaled_age` + `sex_female` +
`biopsy_structure` (biopsyStructure, already collapsed to a low-cardinality
grouping in `3_meta.ipynb`/`helpers/config.r`) + `scaled_purity` + `scaled_tmb`
-- the last added under the **TMB/Clinical/Purity** covariate spec described
below (see "Covariate-adjustment dimension"); the **None** spec adds nothing
at all, matching `5_models.ipynb`/`8_forest.ipynb`/
`20_ici_specificity_main.ipynb`'s own "None" convention. Age, purity, and TMB
are each standardized once across the full analysis population (all ICI and
comparator patients pooled), mirroring `5_models.ipynb`'s single-pooled-
scaling convention; sex and biopsy location are dummy-coded by
`glm()`/`coxph()` directly. A covariate is dropped from a given model only if
it's constant within that particular fitted group (same pattern used for
`cohort_pan`). `scaled_tmb` uses the same `log2(tmbPerMb + 1)` + `scale()`
convention as `5_models.ipynb`/`9_roc.ipynb`/`20_ici_specificity_main.ipynb`.

TMB is used two different ways in this notebook, which is worth keeping
straight: as an adjustment *covariate* (above, only under the TMB/Clinical/
Purity spec) versus, separately and unrelated to that covariate, to *define
the patient subset* for two of the nine ICI-specificity interaction tests
below (TMB High / TMB Low) -- the same way `msi_high` defines two others.

**Covariate-adjustment dimension:** every effect-size and interaction test
below is fit twice, under two covariate specs -- **None** (no covariates at
all) and **TMB/Clinical/Purity** (the full set above) -- shown as separate,
dodged points with distinct shapes (circle = None, triangle = TMB/Clinical/
Purity), the same convention used throughout the paper
(`5_models.ipynb`/`8_forest.ipynb`/`20_ici_specificity_main.ipynb`'s Figure
6). Unlike Figure 6 (which applies this only to its own smaller set of
headline rows), every row here -- including the mechanism sub-groups and the
MSS sensitivity rows -- gets both specs, so the adjustment's impact can be
read consistently across this notebook's full breadth. `drop_na()` always
requires every adjustment-related column regardless of which spec is being
fit, so the fitted population is identical across both specs for a given
group/endpoint -- any difference in the estimate is attributable to the
covariate set, not a different sample.

**Caution on purity specifically:** tumor purity is not a clean, independent
confounder here -- lower purity partly reflects *more* immune infiltration
(the non-tumor fraction PURPLE measures includes infiltrating T cells), which
is exactly what CIDER itself measures. Adjusting for it can therefore
attenuate part of the true CIDER effect rather than purely removing bias, so
a weaker interaction term after this adjustment is not automatically evidence
against specificity. Included here per explicit request; interpret
TMB/Clinical/Purity-adjusted estimates with that in mind, especially
alongside the depth-only (non-purity-adjusted) `cider_score_depth_adj` used
as the exposure throughout, and alongside the matching **None** spec, which
sidesteps the purity question entirely.

**Caution on covariate count generally:** every additional covariate costs
degrees of freedom, and the smallest comparator arms and mechanism sub-groups
are already close to the `MIN_N`/`MIN_EVENTS` floor -- expect some rows to be
skipped or show wider CIs than a simpler adjustment would, especially under
the fuller TMB/Clinical/Purity spec.

**Row structure:** two effect-type sections, matching Figure 6's convention
(`20_ici_specificity_main.ipynb`, Section 7) --
**Cohort-Specific Effects** on top and **Interaction / Relative Effects**
below -- rather than one panel per treatment group as in an earlier version
of this figure:

- **Cohort-Specific Effects** (top) -- every per-group, within-group effect
  size, i.e. whether CIDER has value regardless of treatment (prognostic):
  - Level 1 (top): ICI Therapy | Non-ICI Therapy
  - Level 2 (mid): Chemotherapy | Targeted therapy | Multiple therapy | Hormonal therapy
  - Level 3 (mech): mechanism sub-groups within each level
    - ICI: derived via `ICI_TREATMENT_NORM` → `ICI_MECHANISM_MAP` (config.r)
    - Non-ICI: `mechanism` column from clinical_update.txt
    - Included only if **≥ 15 responders AND ≥ 15 non-responders** (BOR)
    - Ordered by BOR effect size (OR) descending within each parent
  - MSS-restricted sensitivity rows: ICI Therapy and Non-ICI Therapy's own
    main effects, restricted to microsatellite-stable patients -- lets the
    reader see whether each arm's own effect size still holds once MSI-high
    patients (who respond to ICI through a largely CIDER-independent
    mechanism) are excluded.
- **Interaction / Relative Effects** (below, diamond markers, labelled
  "ICI vs. X") -- the direct test of "specificity": nine `cider_log2 *
  is_ici` interaction models, each fit on the pooled ICI + one comparator's
  patients (the four treatment classes, Non-ICI Therapy overall as a
  coarser summary, two MSI-status-restricted repeats of the Non-ICI Therapy
  comparison -- one on microsatellite-stable (MSS) patients only, one on
  MSI-high patients only -- and two TMB-status-restricted repeats of the
  same comparison -- one on TMB-High patients only, one on TMB-Low patients
  only, both arms restricted the same way in each case). The MSI split
  checks that the signal isn't driven by MSI-high tumors' largely
  CIDER-independent response biology; the TMB split checks the analogous
  question for tumor mutational burden, a correlated but distinct genomic
  correlate of ICI response. The plotted value is the **interaction term**
  (`cider_log2:is_ici`), i.e. a ratio-of-ORs (BOR/DCB) or ratio-of-HRs
  (PFS/OS) — how much stronger (or weaker) the CIDER-score effect is in ICI
  patients versus that comparator; the Cohort-Specific Effects rows above
  show the effect size *within* each group, but only the interaction term
  formally tests whether that effect *differs* between ICI and the
  comparator.

**Layout:** `facet_grid(section ~ endpoint)` — just the two effect-type
sections above as row panels (not one panel per treatment group, as in an
earlier version of this figure) — by BOR, DCB, PFS, OS column panels. Small
italic labels ("Cohort-Specific Effects" / "Interaction / Relative Effects")
are inserted directly into the plot's gtable whitespace between and above
the two row-facets, the same technique `20_ici_specificity_main.ipynb` uses
for Figure 6 — there is no native ggplot layer that draws inside
`panel.spacing`. The resulting saved object is therefore a gtable/grob, not
a plain ggplot. Extra vertical spacing is inserted between treatment-class
blocks within each section (e.g. between the ICI Therapy rows and the
Non-ICI Therapy rows) so the original per-class grouping remains visually
readable even without separate facet panels for each.

CIs use the Wald method (±1.96 × SE on the log scale). Tumor-type covariate
cells with < 20 patients are recoded to 'Other' (all patients retained).
Mechanism labels omit the parent prefix — the row label's own indentation
and the surrounding treatment-class rows provide that context.

FDR correction (BH) is applied within each endpoint, across **all** tests
fitted for that endpoint under **both** covariate specs together (every
per-group effect-size test, the MSS sensitivity rows, the mechanism
sub-group tests, and the nine ICI-specificity interaction tests, at both
None and TMB/Clinical/Purity) — one family per endpoint. This FDR family is
specific to this notebook's much larger scope; it is not comparable to
Figure 6's own, smaller 2-spec x 9-group x 2-endpoint family.


In [ ]:
library(dplyr)
library(tidyr)
library(purrr)
library(broom)
library(survival)
library(ggtext)
library(ggplot2)
library(data.table)

In [ ]:
args <- commandArgs(trailingOnly = FALSE)
script_path <- sub("--file=", "", args[grep("--file=", args)])
SCRIPT_DIR <- if (length(script_path) > 0) {
  normalizePath(dirname(script_path), mustWork = FALSE)
} else {
  getwd()
}

source(file.path(SCRIPT_DIR, "../helpers/map.r"))
source(file.path(SCRIPT_DIR, "../helpers/shortcuts.r"))
source(file.path(SCRIPT_DIR, "../helpers/config.r"))   # ICI_TREATMENT_NORM, ICI_MECHANISM_MAP
source(file.path(SCRIPT_DIR, "../helpers/themes.r"))

## 1 — Data preparation

In [ ]:
# Load clinical metadata.
# 'treatment_str' is renamed from 'treatment' to avoid collision with the
# existing 'treatment' column in ready.rds after the join.
clinical_raw <- data.table::fread(paste0(READY_DIR, "clinical_update.txt")) |>
  dplyr::transmute(
    sampleId       = sampleId,
    treatment_type = type,
    treatment_str  = treatment,
    mechanism_raw  = mechanism
  ) |>
  dplyr::distinct(sampleId, .keep_all = TRUE)

message("Treatment type distribution:")
print(sort(table(clinical_raw$treatment_type), decreasing = TRUE))

message("\nNon-ICI mechanism distribution (top 30):")
print(
  clinical_raw |>
    dplyr::filter(!treatment_type %in% c("Immunotherapy", "", "NULL")) |>
    dplyr::count(treatment_type, mechanism_raw, sort = TRUE) |>
    head(30)
)

In [ ]:
# Load full ready object (T-cell group); join clinical metadata.
# ready.rds duplicates every patient: once with the natural cohort_broad and
# once with cohort_broad = 'ICI-Treated' (see 4_base.ipynb bind_rows call).
# Deduplicate by sampleId immediately to avoid inflated Ns.
# Classify non-ICI treatment regimens into mechanism sub-groups based on
# the treatment string. Returns NA for groups too small to be informative
# (e.g. irinotecan alone, ICI-containing combinations, anti-EGFR).
classify_non_ici_mech <- function(trt) {
  if (is.na(trt) || trt == "") return(NA_character_)
  t <- tolower(trt)
  has <- function(...) any(vapply(c(...), function(d) grepl(d, t, fixed = TRUE), logical(1)))
  # Exclude ICI-containing regimens from non-ICI sections
  if (has("pembrolizumab","nivolumab","atezolizumab","ipilimumab",
           "durvalumab","avelumab","tremelimumab")) return(NA_character_)
  # Targeted / biologic add-ons
  if (has("palbociclib","ribociclib","abemaciclib"))              return("CDK4/6i")
  if (has("trastuzumab","pertuzumab","lapatinib","ado-trastuzumab")) return("Chemo + Anti-HER2")
  if (has("bevacizumab","ramucirumab"))                           return("Chemo + Anti-VEGF")
  # Cetuximab/panitumumab: too few responders to model — drop
  if (has("cetuximab","panitumumab"))                             return(NA_character_)
  # Pure chemotherapy sub-groups (ordered from most specific to most general)
  has_plat <- grepl("carboplatin|cisplatin|oxaliplatin", t)
  has_gem  <- grepl("gemcitabine", t)
  has_tax  <- grepl("paclitaxel|docetaxel|cabazitaxel|nab-paclitaxel", t)
  has_fp   <- grepl("capecitabine|fluorouracil|trifluridine|tipiracil", t)
  has_anth <- grepl("doxorubicin|epirubicin", t)
  has_iri  <- grepl("irinotecan", t)
  has_etop <- grepl("etoposide", t)
  if (has_gem  && has_plat && !has_tax && !has_fp)  return("Gemcitabine + Platinum")
  if (has_tax  && has_plat)                         return("Taxane + Platinum")
  if (has_fp   && grepl("oxaliplatin", t) && !has_tax) return("Fluoropyrimidine + Oxaliplatin")
  # Irinotecan alone or with FP: too few responders — drop
  if (has_iri)                                      return(NA_character_)
  if (has_tax  && !has_plat)                        return("Taxane")
  if (has_fp   && !has_plat && !has_anth)           return("Fluoropyrimidine")
  if (has_anth)                                     return("Anthracycline")
  if (has_etop && has_plat)                         return("Etoposide + Platinum")
  if (has_plat && !has_gem && !has_tax && !has_fp)  return("Platinum")
  return(NA_character_)
}

ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::left_join(clinical_raw, by = "sampleId") |>
  dplyr::mutate(
    cider_log2 = log2(cider_score_depth_adj + 1),
    response   = as.integer(response),
    dcb        = as.integer(dcb),
    os_months  = osDays / 30.44,
    pfs_months = pfsEventDays / 30.44,
    cohort_pan = as.character(cohort_pan),
    # Adjustment covariates: age, sex, biopsy location, tumor purity, and
    # TMB. Age, purity, and TMB are each standardized once across this
    # notebook's full analysis population (all ICI + comparator patients
    # pooled), mirroring 5_models.ipynb's single-pooled-scaling convention
    # for covariates that are adjustment terms only (not effects of
    # interest in their own right). biopsyStructure arrives from
    # 3_meta.ipynb already collapsed to BIOPSY_STRUCTURE_LEVELS
    # (helpers/config.r) -- reused as-is here rather than re-grouped, since
    # that collapsing already exists specifically to avoid small
    # biopsy-location cells. See this notebook's intro for a caution on
    # purity specifically -- it's partly a consequence of T-cell
    # infiltration, not purely an independent confounder of it.
    scaled_age       = as.vector(scale(as.numeric(age))),
    sex_female       = as.integer(tolower(sex) %in% c("female", "f")),
    biopsy_structure = biopsyStructure,
    scaled_purity    = as.vector(scale(as.numeric(purity))),
    # Same log2(tmbPerMb + 1) + scale() convention as 5_models.ipynb/
    # 9_roc.ipynb/20_ici_specificity_main.ipynb. Used only as the
    # TMB/Clinical/Purity adjustment covariate below -- NOT the same use as
    # tmb_high, which separately defines the TMB-High/TMB-Low interaction
    # test subsets further down.
    scaled_tmb       = as.vector(scale(log2(tmbPerMb + 1)))
  ) |>
  dplyr::mutate(
    treatment_norm = unname(ICI_TREATMENT_NORM[treatment_str]),
    treatment_norm = dplyr::coalesce(treatment_norm, treatment_str)
  ) |>
  dplyr::mutate(
    ici_mechanism   = unname(ICI_MECHANISM_MAP[treatment_norm]),
    ici_mechanism   = dplyr::coalesce(ici_mechanism, "Other"),
    # ICI patients: use ICI mechanism map.
    # Chemo/Multiple therapy: reclassify by treatment string.
    # Targeted/Hormonal therapy: keep original mechanism_raw label.
    mechanism_group = dplyr::case_when(
      cpi == TRUE ~ ici_mechanism,
      treatment_type %in% c("Chemotherapy", "Multiple therapy") ~
        unname(vapply(treatment_str, classify_non_ici_mech, character(1))),
      TRUE ~ mechanism_raw
    )
  ) |>
  dplyr::mutate(
    # Improve ICI mechanism labels:
    # 'Combo' (CTLA4+PD1 combinations) → descriptive name;
    # 'Other' (ICI patients on non-standard/combination regimens) →
    #   'ICI + Other' so it passes the mechanism filter and appears in the plot.
    mechanism_group = dplyr::case_when(
      cpi == TRUE & mechanism_group == "Combo"  ~ "Anti-CTLA4/PD1",
      cpi == TRUE & mechanism_group == "Other"  ~ "ICI + Other",
      TRUE ~ mechanism_group
    )
  ) |>
  dplyr::select(-treatment_norm, -ici_mechanism) |>
  # Restrict to patients with both response (BOR) and OS annotation,
  # matching the inclusion criteria used in the main analyses.
  dplyr::filter(!is.na(response), !is.na(os_months), !is.na(osEvent))

message("Total unique patients: ", dplyr::n_distinct(ready$sampleId))

message("\nICI mechanism_group distribution:")
print(ready |> dplyr::filter(cpi == TRUE) |> dplyr::count(mechanism_group, sort = TRUE))

for (ttype in c("Chemotherapy", "Targeted therapy", "Hormonal therapy")) {
  message("\n", ttype, " mechanism_group distribution:")
  print(
    ready |>
      dplyr::filter(treatment_type == ttype) |>
      dplyr::count(mechanism_group, sort = TRUE) |>
      head(15)
  )
}

## 2 — Mechanism sub-group definitions

In [ ]:
MIN_POS <- 15
MIN_NEG <- 15

qualifying_mechs <- function(df) {
  df |>
    dplyr::filter(
      !is.na(mechanism_group),
      !mechanism_group %in% c("", "Other", "NULL")
    ) |>
    dplyr::group_by(mechanism_group) |>
    dplyr::summarise(
      n_pos = sum(response == 1L, na.rm = TRUE),
      n_neg = sum(response == 0L, na.rm = TRUE),
      n_tot = dplyr::n(),
      .groups = "drop"
    ) |>
    dplyr::filter(n_pos >= MIN_POS, n_neg >= MIN_NEG) |>
    dplyr::arrange(dplyr::desc(n_tot)) |>
    dplyr::pull(mechanism_group)
}

# Top-level and mid-level groups (used for parent-level model fits).
parent_dfs <- list(
  "ICI Therapy"      = ready |> dplyr::filter(cpi == TRUE) |>
                        dplyr::mutate(cohort_pan = as.character(cohort_broad)),
  "Non-ICI Therapy"  = ready |> dplyr::filter(cpi == FALSE),
  "Chemotherapy"     = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Chemotherapy"),
  "Targeted therapy" = ready |> dplyr::filter(cpi == FALSE,
                          treatment_type == "Targeted therapy" | mechanism_group == "CDK4/6i"),
  "Multiple therapy" = ready |> dplyr::filter(cpi == FALSE,
                          mechanism_group %in% c("Chemo + Anti-HER2", "Chemo + Anti-VEGF")),
  "Hormonal therapy" = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Hormonal therapy")
)

# Mechanism sub-groups — derived only from groups that have a meaningful
# mechanism breakdown: ICI Therapy + the three Non-ICI sub-types.
# 'Non-ICI Therapy' is excluded here because its mechanisms are already
# captured under Chemotherapy / Targeted therapy / Hormonal therapy,
# and including it would produce sub-groups with no entry in GROUP_ORDER.
mech_parent_names <- c("ICI Therapy", "Chemotherapy", "Multiple therapy", "Targeted therapy", "Hormonal therapy")

mech_dfs <- purrr::imap(
  parent_dfs[mech_parent_names],
  function(df, parent_label) {
    # Strip names from mechanism_group to prevent attribute-length mismatches
    df <- dplyr::mutate(df, mechanism_group = unname(mechanism_group))
    mechs <- qualifying_mechs(df)
    message(parent_label, ": ", length(mechs), " qualifying mechanism(s)",
            if (length(mechs)) paste0(" — ", paste(mechs, collapse = ", ")) else "")
    if (length(mechs) == 0L) return(list())
    # Use named vector as input so setNames is not needed
    named_mechs <- stats::setNames(mechs, paste0(parent_label, ": ", mechs))
    purrr::imap(
      named_mechs,
      function(mech, group_name) {
        df |>
          dplyr::filter(mechanism_group == mech) |>
          dplyr::mutate(group = group_name, parent = parent_label)
      }
    )
  }
) |> purrr::flatten()

message("\nTotal mechanism sub-groups: ", length(mech_dfs))

## 3 — Regression functions

In [ ]:
MIN_N        <- 20
MIN_EVENTS   <- 5
# cohort_pan levels with fewer than MIN_COHORT_N patients are recoded to
# 'Other' so all patients are retained while sparse covariate cells are
# collapsed, reducing separation risk in small sub-groups.
MIN_COHORT_N <- 10

# 95% Wald CI helper: avoids profile.glm() convergence failures.
wald_ci_exp <- function(fit, term_name) {
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == term_name) |>
    dplyr::mutate(
      conf.low  = exp(estimate - 1.96 * std.error),
      conf.high = exp(estimate + 1.96 * std.error),
      estimate  = exp(estimate)
    )
}

# Collapse small cohort_pan cells into 'Other' (retains all patients).
recode_small_cohorts <- function(df) {
  df |>
    dplyr::group_by(cohort_pan) |>
    dplyr::mutate(
      cohort_pan = ifelse(dplyr::n() < MIN_COHORT_N, "Other", cohort_pan)
    ) |>
    dplyr::ungroup()
}

# Extra adjustment covariates (age, sex, biopsy location, purity, TMB),
# added on top of cohort_pan. Each is included only if it actually varies
# within the fitted group -- same "drop if constant" pattern already used
# for cohort_pan above -- since a constant covariate in a small comparator
# arm or mechanism sub-group would otherwise waste a degree of freedom or
# block convergence for no informational gain. scaled_tmb is used only
# under the "TMB/Clinical/Purity" spec below (see build_adjustment_covars()
# and this notebook's intro).
extra_adjustment_covariates <- function(df) {
  extra <- character(0)
  if (dplyr::n_distinct(df$scaled_age, na.rm = TRUE) > 1)       extra <- c(extra, "scaled_age")
  if (dplyr::n_distinct(df$sex_female, na.rm = TRUE) > 1)       extra <- c(extra, "sex_female")
  if (dplyr::n_distinct(df$biopsy_structure, na.rm = TRUE) > 1) extra <- c(extra, "biopsy_structure")
  if (dplyr::n_distinct(df$scaled_purity, na.rm = TRUE) > 1)    extra <- c(extra, "scaled_purity")
  if (dplyr::n_distinct(df$scaled_tmb, na.rm = TRUE) > 1)       extra <- c(extra, "scaled_tmb")
  extra
}

# Named covariate-adjustment specs, shared by every fit_* function below via
# the `adjustment` argument (default "Adjusted" everywhere, matching this
# notebook's original always-adjusted behaviour, so any direct call without
# an explicit `adjustment` is unchanged). "Unadjusted" (displayed as "None")
# adds nothing at all -- not even cohort_pan -- same convention as
# 5_models.ipynb's own "None" spec. "Adjusted" (displayed as "TMB/Clinical/
# Purity") is cohort_pan plus every covariate in extra_adjustment_covariates()
# above, i.e. age/sex/biopsy location/purity/TMB wherever each actually
# varies. Same two-spec convention as 20_ici_specificity_main.ipynb's Figure
# 6, applied here to every row rather than just the headline ones.
ADJUSTMENT_LEVELS <- c("Unadjusted", "Adjusted")

build_adjustment_covars <- function(df, adjustment) {
  switch(adjustment,
    "Unadjusted" = character(0),
    "Adjusted" = c(
      if (dplyr::n_distinct(df$cohort_pan) > 1) "cohort_pan",
      extra_adjustment_covariates(df)
    ),
    stop("Unknown adjustment spec: ", adjustment)
  )
}

fit_logistic <- function(df, outcome_col, group_label, endpoint_label,
                         adjustment = "Adjusted") {
  # drop_na() always requires every adjustment-related column (including
  # scaled_tmb), regardless of `adjustment` -- so the fitted population is
  # IDENTICAL across both specs for a given group/endpoint, and any
  # difference in the estimate is attributable to the covariate set, not a
  # different sample.
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(outcome_col)) |>
    recode_small_cohorts()

  outcome <- df[[outcome_col]]
  if (nrow(df) < MIN_N || sum(outcome, na.rm = TRUE) < MIN_EVENTS ||
      length(unique(outcome)) < 2) {
    message("Skipping ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(outcome, na.rm = TRUE),
            ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars_lr <- c("cider_log2", build_adjustment_covars(df, adjustment))
  fit <- tryCatch(
    suppressWarnings(
      glm(reformulate(covars_lr, outcome_col), data = df, family = binomial())
    ),
    error = function(e) NULL
  )
  if (is.null(fit)) return(NULL)
  wald_ci_exp(fit, "cider_log2") |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(outcome, na.rm = TRUE),
                  model = "Logistic", adjustment = adjustment)
}

fit_cox <- function(df, time_col, event_col, group_label, endpoint_label,
                    adjustment = "Adjusted") {
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(c(time_col, event_col))) |>
    dplyr::filter(.data[[time_col]] > 0) |>
    recode_small_cohorts()

  events <- df[[event_col]]
  if (nrow(df) < MIN_N || sum(events, na.rm = TRUE) < MIN_EVENTS) {
    message("Skipping ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(events, na.rm = TRUE),
            ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars_cox <- paste(c("cider_log2", build_adjustment_covars(df, adjustment)), collapse = " + ")
  fmla <- as.formula(paste0("Surv(", time_col, ", ", event_col, ") ~ ", covars_cox))
  fit <- tryCatch(
    suppressWarnings(survival::coxph(fmla, data = df)),
    error = function(e) NULL
  )
  if (is.null(fit)) return(NULL)
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == "cider_log2") |>
    dplyr::mutate(
      conf.low  = exp(estimate - 1.96 * std.error),
      conf.high = exp(estimate + 1.96 * std.error),
      estimate  = exp(estimate)
    ) |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(events, na.rm = TRUE),
                  model = "Cox", adjustment = adjustment)
}

run_all_endpoints <- function(df, group_label, adjustment = "Adjusted") {
  dplyr::bind_rows(
    fit_logistic(df, "response",   group_label, "BOR", adjustment),
    fit_logistic(df, "dcb",        group_label, "DCB", adjustment),
    fit_cox(df, "os_months",  "osEvent",  group_label, "OS",  adjustment),
    fit_cox(df, "pfs_months", "pfsEvent", group_label, "PFS", adjustment)
  )
}

# ── Interaction models (ICI-specificity test) ──────────────────────────────────
# Fits cider_log2 * is_ici on the pooled {ICI Therapy} + {one comparator} data
# and extracts the cider_log2:is_ici term. This is a formal test of whether the
# CIDER-score effect differs between ICI and that comparator (specificity),
# as opposed to fit_logistic()/fit_cox() above which only estimate the effect
# *within* one group at a time.

fit_interaction_logistic <- function(df, outcome_col, group_label, endpoint_label,
                                     adjustment = "Adjusted") {
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, is_ici, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(outcome_col)) |>
    recode_small_cohorts()

  outcome <- df[[outcome_col]]
  if (nrow(df) < MIN_N || sum(outcome, na.rm = TRUE) < MIN_EVENTS ||
      length(unique(outcome)) < 2 || dplyr::n_distinct(df$is_ici) < 2) {
    message("Skipping interaction ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(outcome, na.rm = TRUE),
            ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars <- c("cider_log2 * is_ici", build_adjustment_covars(df, adjustment))
  fit <- tryCatch(
    suppressWarnings(glm(reformulate(covars, outcome_col), data = df, family = binomial())),
    error = function(e) NULL
  )
  if (is.null(fit)) return(NULL)
  wald_ci_exp(fit, "cider_log2:is_ici") |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(outcome, na.rm = TRUE),
                  model = "Logistic (interaction)", adjustment = adjustment)
}

fit_interaction_cox <- function(df, time_col, event_col, group_label, endpoint_label,
                                adjustment = "Adjusted") {
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, is_ici, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(c(time_col, event_col))) |>
    dplyr::filter(.data[[time_col]] > 0) |>
    recode_small_cohorts()

  events <- df[[event_col]]
  if (nrow(df) < MIN_N || sum(events, na.rm = TRUE) < MIN_EVENTS ||
      dplyr::n_distinct(df$is_ici) < 2) {
    message("Skipping interaction ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(events, na.rm = TRUE),
            ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars <- paste(c("cider_log2 * is_ici", build_adjustment_covars(df, adjustment)), collapse = " + ")
  fmla <- as.formula(paste0("Surv(", time_col, ", ", event_col, ") ~ ", covars))
  fit <- tryCatch(suppressWarnings(survival::coxph(fmla, data = df)), error = function(e) NULL)
  if (is.null(fit)) return(NULL)
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == "cider_log2:is_ici") |>
    dplyr::mutate(
      conf.low  = exp(estimate - 1.96 * std.error),
      conf.high = exp(estimate + 1.96 * std.error),
      estimate  = exp(estimate)
    ) |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(events, na.rm = TRUE),
                  model = "Cox (interaction)", adjustment = adjustment)
}

run_interaction_endpoints <- function(df, group_label, adjustment = "Adjusted") {
  dplyr::bind_rows(
    fit_interaction_logistic(df, "response",   group_label, "BOR", adjustment),
    fit_interaction_logistic(df, "dcb",        group_label, "DCB", adjustment),
    fit_interaction_cox(df, "os_months",  "osEvent",  group_label, "OS",  adjustment),
    fit_interaction_cox(df, "pfs_months", "pfsEvent", group_label, "PFS", adjustment)
  )
}

## 4 — Fit models

In [ ]:
# Each of the four result blocks below (parent, MSS-restricted, mechanism,
# interaction) is fit once per ADJUSTMENT_LEVELS spec ("Unadjusted"/
# "Adjusted", defined in the regression-functions cell above) via
# purrr::map_dfr() over the spec, wrapping the original purrr::imap() over
# groups -- same nesting pattern 20_ici_specificity_main.ipynb uses to fit
# its own predprog_parent_results/predprog_interaction_results under both
# specs. This doubles every fitted-model block below; see this notebook's
# intro for why ("Covariate-adjustment dimension").
parent_results <- purrr::map_dfr(
  ADJUSTMENT_LEVELS,
  function(adj) purrr::imap(parent_dfs, ~ run_all_endpoints(.x, .y, adjustment = adj)) |> dplyr::bind_rows()
) |>
  dplyr::mutate(
    group_level = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), "top", "mid"),
    parent      = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), NA_character_, "Non-ICI Therapy")
  )

# Two extra rows: ICI Therapy and Non-ICI Therapy's own main effects,
# restricted to microsatellite-stable (MSS) patients -- the same MSS
# restriction the "Non-ICI (MSS)" interaction test below already uses
# (parent_dfs[["ICI Therapy"/"Non-ICI Therapy"]] |> filter(msi_high == 0)),
# just fit as ordinary within-group effects instead of an interaction term.
# Lets the reader see whether each arm's own effect size still holds once
# MSI-high patients (who respond to ICI through a largely CIDER-independent
# mechanism) are excluded. Originally built for 20_ici_specificity_main.ipynb's
# simplified main-figure companion, moved here since it's a sensitivity/
# robustness check, not part of that figure's headline predictive/
# prognostic message.
mss_parent_dfs <- list(
  "ICI Therapy, MSS"     = parent_dfs[["ICI Therapy"]]     |> dplyr::filter(msi_high == 0),
  "Non-ICI Therapy, MSS" = parent_dfs[["Non-ICI Therapy"]] |> dplyr::filter(msi_high == 0)
)

mss_parent_results <- purrr::map_dfr(
  ADJUSTMENT_LEVELS,
  function(adj) purrr::imap(mss_parent_dfs, ~ run_all_endpoints(.x, .y, adjustment = adj)) |> dplyr::bind_rows()
) |>
  dplyr::mutate(
    group_level = "mss_main",
    parent      = ifelse(group == "ICI Therapy, MSS", "ICI Therapy", "Non-ICI Therapy")
  )

mech_results <- purrr::map_dfr(
  ADJUSTMENT_LEVELS,
  function(adj) {
    purrr::imap(
      mech_dfs,
      ~ run_all_endpoints(.x, .y, adjustment = adj) |>
          dplyr::mutate(group_level = "mech", parent = unique(.x$parent))
    ) |> dplyr::bind_rows()
  }
)

# ── Interaction models (ICI-specificity test) ──────────────────────────────────
# Pool ICI Therapy with each comparator in turn (the four named treatment
# classes, plus Non-ICI Therapy overall as a coarser summary test) and fit
# cider_log2 * is_ici; the interaction term tests whether the CIDER-score
# effect genuinely differs between ICI and that comparator, rather than just
# looking stronger in the ICI row above by eye.
INTERACTION_COMPARATORS <- c("Non-ICI Therapy", "Chemotherapy", "Targeted therapy", "Multiple therapy", "Hormonal therapy")

interaction_dfs <- purrr::map(
  INTERACTION_COMPARATORS,
  function(comparator) {
    dplyr::bind_rows(parent_dfs[["ICI Therapy"]], parent_dfs[[comparator]]) |>
      dplyr::mutate(is_ici = as.integer(cpi == TRUE))
  }
) |> stats::setNames(paste0(INTERACTION_COMPARATORS, "__interaction"))

# Sixth interaction test: the same ICI vs. Non-ICI Therapy comparison above,
# restricted to microsatellite-stable (MSS) patients on both arms. MSI-high
# tumors respond to ICI through a largely CIDER-independent mechanism (high
# neoantigen burden from mismatch-repair deficiency), so pooling MSI-high and
# MSS patients into the headline "Non-ICI Therapy" interaction test above
# could make the ICI arm's CIDER-score effect look stronger than it really is
# for ordinary (MSS) patients. `msi_high == 0` also drops patients with
# unknown MSI status (NA), rather than assuming they are MSS.
interaction_dfs[["Non-ICI (MSS)__interaction"]] <-
  dplyr::bind_rows(
    parent_dfs[["ICI Therapy"]]     |> dplyr::filter(msi_high == 0),
    parent_dfs[["Non-ICI Therapy"]] |> dplyr::filter(msi_high == 0)
  ) |>
  dplyr::mutate(is_ici = as.integer(cpi == TRUE))

# Seventh interaction test: the mirror image of the MSS test above, restricted
# to MSI-high patients on both arms instead. MSI-high ICI patients respond
# very well regardless of CIDER T-cell score (the neoantigen-burden mechanism
# dominates), so this cohort is a genuine stress test of specificity -- if the
# interaction term weakens or flips here, that's expected and informative,
# not a sign the MSS result is wrong. Same NA handling as above: unknown MSI
# status is dropped, not assumed MSI-high.
interaction_dfs[["Non-ICI (MSI)__interaction"]] <-
  dplyr::bind_rows(
    parent_dfs[["ICI Therapy"]]     |> dplyr::filter(msi_high == 1),
    parent_dfs[["Non-ICI Therapy"]] |> dplyr::filter(msi_high == 1)
  ) |>
  dplyr::mutate(is_ici = as.integer(cpi == TRUE))

# Eighth interaction test: the same ICI vs. Non-ICI Therapy comparison above,
# restricted to TMB-High patients on both arms. TMB is a correlated but
# distinct genomic correlate of ICI response (high mutational burden drives
# neoantigen load somewhat independently of T-cell infiltration), so this
# checks whether the specificity signal holds within a fixed TMB stratum
# rather than being partly attributable to a TMB imbalance between the ICI
# and Non-ICI arms. `tmb_high == 1` drops patients with unknown TMB status,
# rather than assuming they are TMB-High. Note this is a *subset* criterion,
# unrelated to scaled_tmb's separate use as an adjustment covariate above.
interaction_dfs[["Non-ICI (TMB High)__interaction"]] <-
  dplyr::bind_rows(
    parent_dfs[["ICI Therapy"]]     |> dplyr::filter(tmb_high == 1),
    parent_dfs[["Non-ICI Therapy"]] |> dplyr::filter(tmb_high == 1)
  ) |>
  dplyr::mutate(is_ici = as.integer(cpi == TRUE))

# Ninth interaction test: the mirror image of the TMB-High test above,
# restricted to TMB-Low patients on both arms instead.
interaction_dfs[["Non-ICI (TMB Low)__interaction"]] <-
  dplyr::bind_rows(
    parent_dfs[["ICI Therapy"]]     |> dplyr::filter(tmb_high == 0),
    parent_dfs[["Non-ICI Therapy"]] |> dplyr::filter(tmb_high == 0)
  ) |>
  dplyr::mutate(is_ici = as.integer(cpi == TRUE))

interaction_results <- purrr::map_dfr(
  ADJUSTMENT_LEVELS,
  function(adj) purrr::imap(interaction_dfs, ~ run_interaction_endpoints(.x, .y, adjustment = adj)) |> dplyr::bind_rows()
) |>
  dplyr::mutate(
    group_level = "interaction",
    parent      = sub("__interaction$", "", group)
  )

# Per-arm sample sizes (N = X ICI + N = Y comparator) for the row labels below,
# computed once from the pooled interaction data rather than per endpoint --
# the split barely moves with endpoint-specific missingness, and one stable
# pair of numbers per test is clearer than four slightly different ones.
# (Also independent of `adjustment` -- the same two arms are pooled either
# way, only the covariate set fit on top of them differs.)
interaction_n_df <- purrr::imap_dfr(
  interaction_dfs,
  function(df, group_name) {
    dplyr::tibble(
      group   = group_name,
      n_ici   = sum(df$is_ici == 1L),
      n_other = sum(df$is_ici == 0L)
    )
  }
)
interaction_results <- interaction_results |>
  dplyr::left_join(interaction_n_df, by = "group")

message("\nInteraction models fitted: ", nrow(interaction_results), " rows across ",
        length(interaction_dfs), " comparisons (", length(INTERACTION_COMPARATORS),
        " treatment-class comparators, plus Non-ICI Therapy restricted to MSS, MSI-high, ",
        "TMB-High, and TMB-Low patients) x ", length(ADJUSTMENT_LEVELS), " adjustment specs")

results <- dplyr::bind_rows(parent_results, mss_parent_results, mech_results, interaction_results) |>
  dplyr::mutate(
    # Level order here drives position_dodge()'s vertical stacking in the
    # forest plot below (first level dodges to the bottom of each row's
    # slot, last level to the top) -- "TMB/Clinical/Purity" first / "None"
    # last puts the None point above the TMB/Clinical/Purity point for
    # every row, same convention as 20_ici_specificity_main.ipynb's Figure
    # 6. Independent of the legend's display order, set separately via
    # scale_shape_manual(breaks = ...) in the plot cell.
    adjustment = factor(
      dplyr::recode(adjustment, "Unadjusted" = "None", "Adjusted" = "TMB/Clinical/Purity"),
      levels = c("TMB/Clinical/Purity", "None")
    )
  )

# ── FDR correction ────────────────────────────────────────────────────────────
# This figure fits a few dozen tests per endpoint (every per-group
# effect-size test, the MSS sensitivity rows, the mechanism sub-group tests,
# and the nine ICI-specificity interaction tests), now doubled across the
# None / TMB/Clinical/Purity covariate specs. Benjamini-Hochberg FDR
# correction is applied across ALL of them together within each endpoint
# (BOR, DCB, PFS, OS are treated as separate test families) -- every model
# fitted for a given endpoint in this figure, at either covariate spec, is
# corrected as one family.
#
# Intervals are left as standard fixed-width 95% Wald CIs (a CI's width
# should reflect a pre-specified coverage probability, not be derived from
# that same test's own p-value — deriving width from p.adj degenerates to a
# near-zero-width interval for non-significant tests). FDR-adjusted p-values
# are used only to flag significance (stars / point opacity below).
results <- results |>
  dplyr::group_by(endpoint) |>
  dplyr::mutate(p.adj = p.adjust(p.value, method = "fdr")) |>
  dplyr::ungroup()

message("\nResults summary:")
print(results |> dplyr::select(group, group_level, endpoint, adjustment, n, n_events,
                                estimate, conf.low, conf.high, p.value, p.adj))


## 5 — Forest plot

In [ ]:
# ── Label lookup ─────────────────────────────────────────────────────────────

group_n <- results |>
  dplyr::group_by(group, group_level, parent) |>
  dplyr::summarise(
    # n/n_events are identical across both adjustment specs (drop_na()
    # always requires every adjustment-related column regardless of spec --
    # see the regression-functions cell), so max() here just collapses the
    # now-duplicated rows down to one value per group, same as before.
    n_total = max(n, na.rm = TRUE),
    # NA for every non-interaction group (n_ici/n_other only exist on
    # interaction_results); first() just carries that through unchanged.
    n_ici   = dplyr::first(n_ici),
    n_other = dplyr::first(n_other),
    .groups = "drop"
  )

mech_order_for <- function(parent_label) {
  # Sort by OS HR ascending (greatest hazard reduction first), using only
  # the TMB/Clinical/Purity spec's estimates -- results now has two rows
  # per (group, endpoint) after the adjustment-spec doubling, so this must
  # pick one spec before arrange()/pull(), or every mechanism would appear
  # twice in the wrong order. TMB/Clinical/Purity (not None) is used here
  # since it's this notebook's primary, fully-adjusted spec -- same one the
  # original (pre-adjustment-dimension) version of this figure always used.
  # Mechanisms without an OS result are appended at the end by N.
  bor_est <- results |>
    dplyr::filter(group_level == "mech", parent == parent_label, endpoint == "OS",
                  adjustment == "TMB/Clinical/Purity") |>
    dplyr::arrange(estimate) |>
    dplyr::pull(group)
  all_mechs <- group_n |>
    dplyr::filter(group_level == "mech", parent == parent_label) |>
    dplyr::arrange(dplyr::desc(n_total)) |>
    dplyr::pull(group)
  c(bor_est, setdiff(all_mechs, bor_est))
}

# The two effect-type row-facets, matching 20_ici_specificity_main.ipynb's
# Figure 6 convention exactly (see this notebook's intro). Everything that
# isn't one of the nine interaction tests is a Cohort-Specific Effect.
SECTION_LEVELS <- c("Cohort-Specific Effect", "Interaction / Relative Effect")

# GROUP_ORDER still lists every row top-to-bottom within its eventual
# section (every interaction test first, then every per-group/mechanism
# row) -- unchanged from this figure's original design. What changed is
# that these are no longer two of seven separate facet panels; `section`
# below collapses them to the two effect-type panels, and `block_for_group()`
# further down (used only for inter-row spacing, not faceting) recovers the
# original seven-way treatment-class grouping so the figure still reads as
# visually organized by treatment class within the single Cohort-Specific
# Effect panel.
GROUP_ORDER <- c(
  "Non-ICI Therapy__interaction",
  "Non-ICI (MSS)__interaction",
  "Non-ICI (MSI)__interaction",
  "Non-ICI (TMB High)__interaction",
  "Non-ICI (TMB Low)__interaction",
  "Chemotherapy__interaction",
  "Multiple therapy__interaction",
  "Targeted therapy__interaction",
  "Hormonal therapy__interaction",
  "ICI Therapy", "ICI Therapy, MSS",
  mech_order_for("ICI Therapy"),
  "Non-ICI Therapy", "Non-ICI Therapy, MSS",
  "Chemotherapy",
  mech_order_for("Chemotherapy"),
  "Multiple therapy",
  mech_order_for("Multiple therapy"),
  "Targeted therapy",
  mech_order_for("Targeted therapy"),
  "Hormonal therapy",
  mech_order_for("Hormonal therapy")
)

label_df <- group_n |>
  dplyr::mutate(
    indent       = ifelse(group_level == "mech", "   ", ""),
    # Mechanism sub-groups: strip the "Parent: " prefix -- with the old
    # per-treatment-class facet panels gone, the row's own position among
    # its neighbours (and the indentation) is what signals "this is a
    # mechanism sub-group of the row above it" now.
    # Interaction rows: name the comparator explicitly -- the section strip
    # text is hidden in this theme (strip.text.y is blank below), so the row
    # label is the only place this appears -- it must be self-explanatory on
    # its own, not just "Interaction" with the comparator implied by context.
    display_name = dplyr::case_when(
      group_level == "interaction" ~ paste0("ICI vs. ", sub(" [Tt]herapy$", "", parent)),
      group_level == "mech"        ~ sub("^[^:]+: ", "", group),
      TRUE                          ~ group
    ),
    display_name = dplyr::case_when(
      display_name == "Fluoropyrimidine + Oxaliplatin" ~ "CAPOX/FOLFOX",
      TRUE ~ display_name
    ),
    # Interaction rows show the two arms' sizes as "N = X vs. Y" -- order
    # matches "ICI vs. <comparator>" in display_name (ICI arm first), so a
    # bare second number is unambiguous without repeating the comparator's
    # name. Same format() + big.mark convention as
    # 20_ici_specificity_main.ipynb's Figure 6 (format(..., big.mark = ",",
    # trim = TRUE) -- trim = TRUE matters here since format() on a whole
    # column pads to a common width by default, which would otherwise leave
    # stray leading spaces inside paste0()).
    group_label  = dplyr::case_when(
      group_level == "interaction" ~ paste0(
        indent, display_name,
        " (N = ", format(n_ici, big.mark = ",", trim = TRUE),
        " vs. ", format(n_other, big.mark = ",", trim = TRUE), ")"
      ),
      TRUE ~ paste0(indent, display_name, " (N = ", format(n_total, big.mark = ",", trim = TRUE), ")")
    )
  )

label_order <- label_df |>
  dplyr::filter(group %in% GROUP_ORDER) |>
  dplyr::mutate(row_order = match(group, GROUP_ORDER)) |>
  dplyr::arrange(row_order) |>
  dplyr::pull(group_label) |>
  rev()

ENDPOINT_LEVELS <- c("BOR", "DCB", "PFS", "OS")

# Recovers the original seven-way treatment-class grouping (ICI Specificity/
# ICI Therapy/Non-ICI Therapy/Chemo/Multiple/Targeted/Hormonal), used below
# only to decide where to widen the gap between rows -- NOT as a facet
# variable anymore (that's `section`'s job now, with just the two
# Cohort-Specific/Interaction levels). Keeping this as its own helper
# function (rather than inlining it) makes clear it's a pure labelling
# convenience, not part of the plot's actual row/column structure.
block_for_group <- function(group, group_level, parent) {
  dplyr::case_when(
    group_level == "interaction" & grepl("^Non-ICI", group) ~ "Non-ICI (interaction family)",
    group_level == "interaction"                            ~ paste0(parent, " (interaction)"),
    group == "ICI Therapy" | group == "ICI Therapy, MSS" |
      (group_level == "mech" & parent == "ICI Therapy")      ~ "ICI Therapy",
    group == "Non-ICI Therapy" | group == "Non-ICI Therapy, MSS" ~ "Non-ICI Therapy",
    group_level == "mech"                                   ~ parent,
    TRUE                                                      ~ group
  )
}

# ── Build plot data ───────────────────────────────────────────────────────────

forest_df <- results |>
  dplyr::left_join(
    label_df |> dplyr::select(group, group_level, group_label),
    by = c("group", "group_level")
  ) |>
  dplyr::mutate(
    group_label = factor(group_label, levels = label_order),
    endpoint    = factor(endpoint, levels = ENDPOINT_LEVELS),
    # Just two row-facets now -- every interaction test together, every
    # per-group (cohort-specific) effect together -- matching Figure 6's
    # convention instead of one panel per treatment group.
    section     = factor(
      ifelse(group_level == "interaction", "Interaction / Relative Effect", "Cohort-Specific Effect"),
      levels = SECTION_LEVELS
    ),
    block       = block_for_group(group, group_level, parent),
    pt_alpha  = ifelse(!is.na(p.adj) & p.adj < 0.1, 1.0, 0.5),
    sig_star  = dplyr::case_when(
      !is.na(p.adj) & p.adj < 0.001 ~ "***",
      !is.na(p.adj) & p.adj < 0.01  ~ "**",
      !is.na(p.adj) & p.adj < 0.1   ~ "*",
      TRUE                          ~ ""
    )
  )

# ── Debug check ───────────────────────────────────────────────────────────────

na_groups <- forest_df |> dplyr::filter(is.na(group_label)) |> dplyr::distinct(group)
if (nrow(na_groups) > 0) {
  warning("Groups with NA label (not in GROUP_ORDER): ",
          paste(na_groups$group, collapse = ", "))
}

plot_df <- forest_df |> dplyr::filter(!is.na(group_label))

# ── Debug figure: raw group strings, no factoring ────────────────────────────
# Plots every fitted group in model order (TMB/Clinical/Purity spec only --
# None would just duplicate every row) -- use to verify nothing is dropped.

debug_df <- forest_df |>
  dplyr::filter(endpoint == "BOR", adjustment == "TMB/Clinical/Purity") |>
  dplyr::mutate(
    debug_label = paste0(
      "[", group_level, "] ", group,
      ifelse(is.na(group_label), " <<< NA LABEL", "")
    )
  ) |>
  dplyr::mutate(debug_label = factor(debug_label, levels = rev(unique(debug_label))))

p_debug <- ggplot2::ggplot(
  debug_df,
  ggplot2::aes(x = estimate, y = debug_label, colour = group_level)
) +
  ggplot2::geom_vline(xintercept = 1, linetype = "dashed", colour = COL_NEGATIVE) +
  ggplot2::geom_pointrange(
    ggplot2::aes(xmin = conf.low, xmax = conf.high), shape = 16, fatten = 2
  ) +
  ggplot2::scale_x_log10() +
  go_theme +
  ggplot2::labs(
    x = "OR (BOR, TMB/Clinical/Purity spec, log scale)", y = NULL, colour = "Level",
    title = "DEBUG — all fitted groups (BOR, TMB/Clinical/Purity spec only, raw labels, no factor ordering)",
    subtitle = "Groups marked <<< NA LABEL are fitted but absent from GROUP_ORDER / label_order"
  )

n_debug <- nrow(debug_df)
set_plot_dims(11, max(4, n_debug * 0.35 + 1.5)); p_debug

ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_ici_specificity_debug.png"),
       p_debug, width = 11, height = max(4, n_debug * 0.35 + 1.5), dpi = 200)
message("Saved debug figure")

# ── Main forest plot ──────────────────────────────────────────────────────────

adaptive_breaks <- function(x) {
  if (max(x, na.rm = TRUE) > 2) c(0.5, 1, 2) else c(0.7, 1, 1.3)
}
# Adaptive x-axis breaks: wide for BOR/DCB (OR), tight for PFS/OS (HR)

# Numeric y position instead of relying on group_label's plain discrete
# factor -- same device 20_ici_specificity_main.ipynb's Figure 6 uses for
# its own y_pos, extended here to insert a widened gap every time the
# underlying treatment-class `block` changes (not just once, above "ICI
# Therapy", since this figure has many more blocks: ICI Therapy, Non-ICI
# Therapy, Chemotherapy, Multiple therapy, Targeted therapy, Hormonal
# therapy within Cohort-Specific Effect, and the Non-ICI interaction family
# vs. each named treatment-class interaction test within Interaction /
# Relative Effect). Without these gaps, the single Cohort-Specific Effect
# panel would read as one undifferentiated stack of ~30 rows now that the
# old per-treatment-class facet panels are gone.
y_lookup <- plot_df |>
  dplyr::distinct(section, group, group_label, block) |>
  dplyr::group_by(section) |>
  dplyr::arrange(group_label, .by_group = TRUE) |>
  dplyr::mutate(
    block_changed = block != dplyr::lag(block, default = dplyr::first(block)),
    gap           = cumsum(block_changed) * 0.3,
    y_pos         = seq_len(dplyr::n()) + gap
  ) |>
  dplyr::ungroup() |>
  dplyr::select(-block_changed, -gap, -block)

# Large additive offset on the Interaction / Relative Effect section's y
# positions, purely to keep the two sections' numeric ranges from
# overlapping -- facet_grid(scales = "free") computes each row-facet
# panel's range independently and only draws the breaks inside its own
# range, so this offset is invisible in the rendered figure (same device
# Figure 6 uses, PREDPROG_SECTION_Y_OFFSET).
SECTION_Y_OFFSET <- c("Cohort-Specific Effect" = 0, "Interaction / Relative Effect" = 1000)
y_lookup <- y_lookup |>
  dplyr::mutate(y_pos = y_pos + SECTION_Y_OFFSET[as.character(section)])

plot_df <- plot_df |>
  dplyr::left_join(y_lookup |> dplyr::select(section, group, group_label, y_pos),
                    by = c("section", "group", "group_label"))

# orientation = "y" makes the dodge direction explicit -- needed now that y
# is a plain numeric column rather than a factor ggplot could auto-detect
# as the categorical/dodge axis. Dodges the None / TMB/Clinical/Purity
# points within each row.
DODGE <- ggplot2::position_dodge(width = 0.5, orientation = "y")

# Symmetric x-axis around 1 (the null effect) per facet panel, on the log
# scale, same device as Figure 6's predprog_xrange_df -- extends each
# (section, endpoint) panel's data with two invisible phantom points via
# geom_blank() so 1 sits exactly in the visual centre instead of wherever
# the data's own min/max CI happen to fall.
xrange_df <- plot_df |>
  dplyr::filter(is.finite(conf.low), is.finite(conf.high), conf.low > 0, conf.high > 0) |>
  dplyr::group_by(section, endpoint) |>
  dplyr::summarise(
    log_bound = max(abs(log10(conf.low)), abs(log10(conf.high))) * 1.05,
    .groups = "drop"
  )
xrange_df <- dplyr::bind_rows(
  xrange_df |> dplyr::mutate(estimate =  10 ^ log_bound),
  xrange_df |> dplyr::mutate(estimate = 10 ^ -log_bound)
)

p_forest <- ggplot2::ggplot(
  plot_df,
  ggplot2::aes(x = estimate, y = y_pos, alpha = pt_alpha, shape = adjustment, colour = section)
) +
  ggplot2::geom_vline(xintercept = 1, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.3) +
  ggplot2::geom_blank(data = xrange_df, ggplot2::aes(x = estimate), inherit.aes = FALSE) +
  # Cohort-Specific Effect rows.
  ggplot2::geom_pointrange(
    data = dplyr::filter(plot_df, group_level != "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    fatten = 2.6, size = 0.42,
    position = DODGE
  ) +
  # Interaction / Relative Effect rows (ICI-specificity test): same accent
  # colour as Figure 6, slightly larger point.
  ggplot2::geom_pointrange(
    data = dplyr::filter(plot_df, group_level == "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    fatten = 3.0, size = 0.5,
    position = DODGE
  ) +
  ggplot2::geom_text(
    ggplot2::aes(x = conf.high, label = sig_star),
    position = DODGE,
    hjust = -0.2, size = 2.6, colour = COL_INK, show.legend = FALSE
  ) +
  # Colour maps to section (Effect Type), same two colours as Figure 6:
  # Cohort-Specific Effect -> COL_NEGATIVE (the baseline/reference per-group
  # reading), Interaction / Relative Effect -> COL_ACCENT (the comparative
  # interaction test).
  ggplot2::scale_colour_manual(
    values = c("Cohort-Specific Effect" = COL_NEGATIVE, "Interaction / Relative Effect" = COL_ACCENT),
    name   = "Effect Type"
  ) +
  # Shape now encodes the covariate-adjustment spec -- same convention and
  # same shapes as Figure 6 and the main forest-plot supplement
  # (8_forest.ipynb): circle for None, triangle for TMB/Clinical/Purity.
  # breaks fixes the legend's display order independent of the adjustment
  # factor's level order (reversed on purpose above, to control dodge
  # stacking instead).
  ggplot2::scale_shape_manual(
    values = c("None" = 16, "TMB/Clinical/Purity" = 17),
    breaks = c("None", "TMB/Clinical/Purity"),
    name   = "Covariates"
  ) +
  # section (rows) x endpoint (columns) -- panel height scales with row
  # count; space = "free_y" keeps the shorter Interaction / Relative Effect
  # block from being stretched to match the taller Cohort-Specific Effect
  # block.
  ggplot2::facet_grid(
    section ~ endpoint,
    scales = "free",
    space  = "free_y",
    labeller = labeller(
      endpoint = c(
        "BOR" = "<b style='font-size:10pt'>BOR</b><br><span style='font-size:7pt; color:grey55'>Worse ←  → Better</span>",
        "DCB" = "<b style='font-size:10pt'>DCB</b><br><span style='font-size:7pt; color:grey55'>Worse ←  → Better</span>",
        "PFS" = "<b style='font-size:10pt'>PFS</b><br><span style='font-size:7pt; color:grey55'>Better ←  → Worse</span>",
        "OS"  = "<b style='font-size:10pt'>OS</b><br><span style='font-size:7pt; color:grey55'>Better ←  → Worse</span>"
      )
    )
  ) +
  ggplot2::scale_alpha_identity() +
  ggplot2::scale_x_log10(breaks = adaptive_breaks,
                         labels = scales::label_number(accuracy = 0.1, drop0trailing = TRUE)) +
  # Breaks/labels come from the same lookup used to build y_pos above --
  # each facet panel only renders the subset of breaks that fall inside its
  # own (free-scaled) range, so this single global scale still shows the
  # correct labels per section despite the shared numbering.
  ggplot2::scale_y_continuous(
    breaks = y_lookup$y_pos,
    labels = as.character(y_lookup$group_label),
    expand = ggplot2::expansion(add = 0.6)
  ) +
  go_theme +
  ggplot2::theme(
    panel.grid.major.y = ggplot2::element_blank(),
    panel.grid.minor.x = ggplot2::element_blank(),
    # Column strips (endpoints)
    strip.text.x        = ggtext::element_markdown(size = 8, halign = 0.5),
    # No section headers -- blank the row strip entirely (text +
    # background); the gap between facet rows is widened below
    # (panel.spacing.y) to fit the small italic text labels dropped
    # directly into that whitespace band via the gtable step after this
    # plot is built (see this notebook's intro / Figure 6's own Section 7).
    strip.text.y       = ggplot2::element_blank(),
    strip.background.y = ggplot2::element_blank(),
    panel.spacing.y    = ggplot2::unit(0.9, "lines"),
    legend.position     = "bottom",
    # Two distinct legends (Effect Type colour, Covariates shape) stacked
    # vertically, Effect Type on top -- matches Figure 6's forest plot.
    legend.box          = "vertical",
    legend.box.just     = "left",
    legend.spacing.y    = ggplot2::unit(0, "pt"),
    legend.margin       = ggplot2::margin(t = 0, b = 0)
  ) +
  ggplot2::guides(
    colour = ggplot2::guide_legend(order = 1),
    shape  = ggplot2::guide_legend(order = 2)
  ) +
  ggplot2::labs(
    x        = "OR / HR (95% CI, log scale)",
    y        = NULL,
    title    = "CIDER T-cell Score: Strongest Signal in ICI Therapies",
    caption  = "Stars: * FDR < 0.1  ** FDR < 0.01  *** FDR < 0.001 (BH-adjusted across all tests and both covariate specs within each endpoint)."
  )

# ── Drop text labels into the gaps above/between the two row-sections ────────
# Same gtable-splicing technique as 20_ici_specificity_main.ipynb's Figure 6
# (Section 7) -- there is no ggplot layer that can draw inside
# panel.spacing, since it's pure gtable whitespace outside any panel's
# coordinate space. The gap *between* the two row-facets already exists
# (from the panel.spacing.y theme setting above); the gap *above* the first
# row-facet has to be spliced in as a brand-new gtable row first, since
# panel.spacing only inserts whitespace *between* facet panels, not above
# the first or below the last. Both labels span the full panel width (every
# endpoint column), not just one.
forest_grob <- ggplot2::ggplotGrob(p_forest)

panel_grobs   <- grep("^panel-", forest_grob$layout$name)
panel_row_idx <- sort(unique(forest_grob$layout$t[panel_grobs]))
stopifnot(length(panel_row_idx) == 2)  # exactly 2 row-facets expected
gap_row_idx   <- panel_row_idx[1] + 1
panel_col_idx <- range(forest_grob$layout$l[panel_grobs])

forest_grob <- gtable::gtable_add_grob(
  forest_grob,
  grid::textGrob(
    "Interaction / Relative Effects",
    gp = grid::gpar(fontsize = 8, fontface = "italic", col = COL_NEGATIVE)
  ),
  t = gap_row_idx, b = gap_row_idx, l = panel_col_idx[1], r = panel_col_idx[2],
  name = "ici-spec-gap-label"
)

# `pos` is 0-indexed from the top, so `panel_row_idx[1] - 1` places the new
# row immediately before the (original) top panel row; gtable_add_rows()
# shifts every row at or below that point down by one and updates $layout
# automatically, so the already-placed bottom-gap label above doesn't need
# to be touched. After the insertion, the new row's own index equals the
# top panel row's *original* index, since that's exactly where it was
# spliced in.
forest_grob <- gtable::gtable_add_rows(
  forest_grob, heights = ggplot2::unit(0.9, "lines"), pos = panel_row_idx[1] - 1
)
forest_grob <- gtable::gtable_add_grob(
  forest_grob,
  grid::textGrob(
    "Cohort-Specific Effects",
    gp = grid::gpar(fontsize = 8, fontface = "italic", col = COL_NEGATIVE)
  ),
  t = panel_row_idx[1], b = panel_row_idx[1], l = panel_col_idx[1], r = panel_col_idx[2],
  name = "ici-spec-top-gap-label"
)

n_rows <- dplyr::n_distinct(plot_df$group_label, na.rm = TRUE)
# 25% shorter per request -- applied as a flat multiplier on the whole
# height formula (including its floor) rather than just tightening the
# per-row spacing, so the row-to-row gap and the floor shrink together in
# the same proportion.
fig_h  <- max(4.5, n_rows * 0.33 + 2.0) * 0.75
set_plot_dims(11, fig_h)
grid::grid.newpage(); grid::grid.draw(forest_grob)

saveRDS(forest_grob, paste0(CIDER_PROJ_DIR, "figs/supplement_ici_specificity.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/supplement_ici_specificity.png"),
       forest_grob, width = 11, height = fig_h, dpi = 300)
message("Saved supplement_ici_specificity.png")
